## Xây dựng Tree Node

In [6]:
import numpy as np
import pandas as pd

class TreeNode(object) :
    def __init__(self, ids = None, children = [], entropy = 0, depth = 0):
        self.ids = ids
        self.entropy = entropy
        self.depth = depth
        self.split_attribute = None
        self.children = children
        self.order = None
        self.label = None
    # Hàm gọi khi qđ đây là nút phân nhánh được và cập nhật thuộc tính
    def set_properties(self, split_attribute, order) :
        self.split_attribute = split_attribute
        self.order = order
    #Hàm gọi khi tt qđ đây là nút lá sau đó gán nhãn
    def set_label(self, label):
        self.label = label

In [7]:
def entropy(freq) :
    # remove prob 0
    freq_0 = freq[np.array(freq).nonzero()[0]] # convert sang array để dùng nonzero hoặc freq_0 = freq[freq != 0
    prob_0 = freq_0/float(freq_0.sum()) #np.sum() sd vectorlization dùng C implement còn sum(freq) dùng for loop
    return -np.sum(prob_0*np.log(prob_0))

In [8]:
class DecisionTreeID3(object):
    #min_gain : nếu Gain quá nhỏ thì k cần chia
    def __init__(self, max_depth = 10, min_samples_split = 2, min_gain = 1e-4):
        self.root = None
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.Ntrain = 0 # Lưu số lượng dữ liệu train
        self.min_gain = min_gain
    
    def fit(self, data, target):
        self.Ntrain = data.shape[0]
        self.data = data
        self.attributes = list(data)
        self.target = target
        self.labels = target.unique()

        ids = range(self.Ntrain) # Tạo danh sách chỉ số từ 0 đến Ntrain-1
        self.root = TreeNode(ids = ids, entropy = self._entropy(ids), depth = 0)
        queue = [self.root]
        while queue:
            node = queue.pop()
            if node.depth < self.max_depth or node.entropy < self.min_gain:
                node.children = self._split(node) # Gọi hàm chia nhánh
                if not node.children: # leaf node
                    self._set_label(node) # Biến nó thành nút lá và gán nhãn
                queue += node.children
            else:
                self._set_label(node)
    def _entropy(self, ids):
        # calculate entropy of a node with index ids
        if (len(ids) == 0): return 0
        ids = [i+1 for i in ids] # pandas index start from 1
        freq = np.array(self.target[ids].value_counts())
        return entropy(freq)
    def _set_label(self, node):
        # find label for a node if it is a leaf
        target_ids = [i+1 for i in node.ids] # target is a series variable
        node.set_label(self.target[target_ids].mode()[0])  # most frequent label
    def _split(self, node):
        ids = node.ids
        best_gain = 0
        best_splits = []
        best_attribute = None
        order = None
        sub_data = self.data.iloc[ids, :]
        for i, att in enumerate(self.attributes):
            values = self.data.iloc[ids, i].unique().tolist()
            if len(values) == 1: continue # entropy = 0
            splits = []
            for val in values: 
                sub_ids = sub_data.index[sub_data[att] == val].tolist()
                splits.append([sub_id-1 for sub_id in sub_ids])
            # don't split if a node has too small number of points
            if min(map(len, splits)) < self.min_samples_split: continue
            # information gain
            HxS= 0
            for split in splits:
                HxS += len(split)*self._entropy(split)/len(ids)
            gain = node.entropy - HxS 
            if gain < self.min_gain: continue # stop if small gain 
            if gain > best_gain:
                best_gain = gain 
                best_splits = splits
                best_attribute = att
                order = values
        node.set_properties(best_attribute, order)
        child_nodes = [TreeNode(ids = split,
                    entropy = self._entropy(split), depth = node.depth + 1) for split in best_splits]
        return child_nodes

    def predict(self, new_data):
        """
        :param new_data: a new dataframe, each row is a datapoint
        :return: predicted labels for each row
        """
        npoints = new_data.count()[0]
        labels = [None]*npoints
        for n in range(npoints):
            x = new_data.iloc[n, :] # one point 
            # start from root and recursively travel if not meet a leaf 
            node = self.root
            while node.children: 
                node = node.children[node.order.index(x[node.split_attribute])]
            labels[n] = node.label
            
        return labels

if __name__ == "__main__":
    df = pd.read_csv(r'D:\ML\machine_learning_homework\data\raw\weather.csv', index_col = 0, parse_dates = True)
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    tree = DecisionTreeID3(max_depth = 3, min_samples_split = 2)
    tree.fit(X, y)
    print(tree.predict(X))
        

['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no']


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19616\309480493.py:89: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(r'D:\ML\machine_learning_homework\data\raw\weather.csv', index_col = 0, parse_dates = True)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19616\309480493.py:76: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  npoints = new_data.count()[0]


In [6]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data = load_iris()
X = data.data
y = data.target

#random_state: Đảm bảo kết quả giống nhau mỗi khi bạn chạy lại mã
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 50)

#criterion đo lường sự phân chia 'gini' hoặc 'entropy'
#max_depth: Độ sâu tối đa của cây. Nếu để cây quá sâu, nó sẽ học thuộc lòng dữ liệu (overfitting)
model = DecisionTreeClassifier(criterion='entropy', max_depth=4)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print (f"Accuracy: {100*accuracy_score(y_test, y_pred):.2f}%")

Accuracy: 96.67%
